# forensic-synth — lanceur Colab
Duplique ce notebook par job/compte. Change seulement **CONFIG** et **STAGE** ci-dessous.
Tout est sauvé sur Drive et **reprenable** après coupure.

Ordre du pipeline et dépendances entre étages :
1. `partition` — aucune dépendance (charge data/blocks.json, valide contre scface_root).
2. `train_generator` — dépend de `partition` (paires réelles du Bloc A).
3. `generate` — dépend du checkpoint de `train_generator` (génère depuis les mugshots du Bloc B).
4. `fidelity` — dépend de `generate` (compare synthétique vs réel, Bloc B). Garde-fou go/no-go.
5. `train_recognition` — dépend de `generate` si condition `synthetic`/`mixed` (sinon juste `partition` pour la condition `real`).
6. `evaluate` — dépend des checkpoints de `train_recognition` (rank-1 sur Bloc C, vierge).


In [ ]:
# 1) Monter Google Drive (mémoire persistante)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) Parametres du job  ---  A MODIFIER par compte/session
CONFIG = 'configs/visible_d1.yaml'   # plus tard : configs/ir_d1.yaml, etc.
STAGE  = 'smoke'                     # smoke|partition|train_generator|generate|fidelity|train_recognition|evaluate
REPO_URL = 'https://github.com/MarouaneAyech/foresynth_facerecog.git'
CODE_DIR = '/content/foresynth_facerecog'         # checkout du code (stockage local VM, rapide, ephemere)
DATA_ROOT = '/content/drive/MyDrive/forensic_synth'  # racine des DONNEES/artefacts (Drive, persistante)


In [ ]:
# 3) Recuperer/mettre a jour le CODE (toujours dans CODE_DIR, separe des donnees)
# Code et donnees sont VOLONTAIREMENT separes : cloner/pull du code directement sur Drive
# est lent et fragile (beaucoup de petits fichiers), et un dossier Drive existant (donnees)
# n'est pas forcement un repo git -> 'git pull' y echouerait. CODE_DIR est recree a chaque
# session (stockage local de la VM, perdu a la coupure, sans consequence : c'est juste du code).
import os, shutil, subprocess

# Se placer hors de CODE_DIR AVANT toute suppression : si une execution precedente de cette
# cellule a deja fait os.chdir(CODE_DIR), rmtree(CODE_DIR) supprimerait le cwd du processus
# lui-meme (-> 'fatal: Unable to read current working directory' sur tout subprocess ensuite).
os.chdir('/content')

if os.path.exists(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', '-C', CODE_DIR, 'pull'], check=True)
else:
    if os.path.exists(CODE_DIR):
        shutil.rmtree(CODE_DIR)
    subprocess.run(['git', 'clone', REPO_URL, CODE_DIR], check=True)
os.chdir(CODE_DIR)

# Racine unique des artefacts/donnees (cf. src/config.py) : seul levier a toucher pour
# changer d'environnement (cf. CLAUDE.md). Reste sur Drive, independant de CODE_DIR.
os.environ['FORENSIC_SYNTH_ROOT'] = DATA_ROOT
# Reduit la fragmentation memoire CUDA (suggere par le message d'erreur OOM
# rencontre en train_generator : VAE decode + ArcFace avec gradients en plus du UNet).
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Dossier code :', os.getcwd())
print('Racine donnees (FORENSIC_SYNTH_ROOT) :', DATA_ROOT)


In [ ]:
# 4) Dependances (figer les versions une fois stables)
!pip -q install -r requirements.txt

# Le depot foivospar/Arc2Face n'a PAS de setup.py/pyproject.toml -> pip install
# git+... echoue silencieusement (pas d'erreur visible avec -q) et 'import arc2face'
# plante plus tard. On le clone a part et on l'ajoute au PYTHONPATH (herite par les
# sous-processus '!python -m experiments.run ...').
import os, subprocess
ARC2FACE_LIB_DIR = '/content/Arc2Face_lib'
if not os.path.exists(os.path.join(ARC2FACE_LIB_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/foivospar/Arc2Face.git', ARC2FACE_LIB_DIR], check=True)
os.environ['PYTHONPATH'] = ARC2FACE_LIB_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH :', os.environ['PYTHONPATH'])


In [ ]:
# 5) Vérifier le GPU (les étages lourds en ont besoin ; smoke/partition non)
import torch
print('CUDA dispo :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


### Assets pre-entraines (deja deposes sous DATA_ROOT/pretrained/)
Structure attendue (cf. `configs/base.yaml`, `paths.pretrained_dir`) :
```
forensic_synth/pretrained/
  arcface/arcface_ms1mv3_r50.pth   (IResNet-50 / ArcFace MS1MV3)
  encoder/config.json, pytorch_model.bin     (Arc2Face, encodeur CLIP modifie)
  arc2face/config.json, diffusion_pytorch_model.safetensors  (Arc2Face, UNet)
  antelopev2/*.onnx                 (insightface, extraction embedding ID)
```
Le code (`src/generator/base_arc2face.py`) utilise ces fichiers directement depuis
Drive en priorite (pas de retelechargement, pas de duplication -- `antelopev2` est
lie par symlink vers `~/.insightface/models/antelopev2` au premier chargement).
Si absents, repli automatique sur le Hub HuggingFace pour Arc2Face (plus lent) ;
pas de repli pour `antelopev2`/`arcface_ms1mv3` (pas auto-telechargeables).


In [ ]:
# 6) Verifier les assets pre-entraines (deja deposes sous DATA_ROOT/pretrained/,
#    cf. cellule precedente -- aucun telechargement ici, juste une verification).
from pathlib import Path
from src.config import load_config

_cfg = load_config(CONFIG)
_checks = {
    'Poids ArcFace iresnet50/arcface_ms1mv3 (generator/identity_loss, fidelity, recognition)':
        Path(_cfg['paths']['arcface_weights']),
    'Modeles Arc2Face (encoder/ + arc2face/, sinon repli HF Hub)':
        Path(_cfg['paths']['arc2face_models_dir']),
    "Pack insightface 'antelopev2' (extraction de l'embedding ID)":
        Path(_cfg['paths']['insightface_models_dir']) / 'antelopev2',
}
for label, p in _checks.items():
    status = 'OK' if p.exists() else 'MANQUANT'
    print(f'[{status}] {label} -> {p}')


In [ ]:
# 7) (Optionnel) câblage d'abord — gratuit, sans GPU
!python -m pytest -q


In [ ]:
# 8) Lancer l'étage demandé (reprend automatiquement sur checkpoint)
!python -m experiments.run --config $CONFIG --stage $STAGE


## Répartition multi-comptes (jobs INDÉPENDANTS)
- Compte 1 : `STAGE='partition'` puis `STAGE='train_generator'` (lourd, 1 fois) → dataset caché sur Drive.
- Compte 1 (suite) : `STAGE='generate'` puis `STAGE='fidelity'` (go/no-go avant de continuer).
- Comptes 2/3/4 : `STAGE='train_recognition'` pour des seeds/conditions différents (jobs séparés, modifie `cfg['recognition']['conditions']`/`cfg['seeds']` ou lance plusieurs notebooks).
- Puis `STAGE='evaluate'` (agrège automatiquement les seeds disponibles par condition).

_Aucune fusion automatique entre comptes : tu ouvres chaque session manuellement._
